# Taller: Análisis exploratorio de datos 

Dataset de apoyo: Insurance (Kaggle).
Este notebook espera el archivo en `01_Taller/data/insurance.csv`.

In [ ]:
# ── Detección de entorno y configuración ──────────────────────────────────
import sys, subprocess

EN_COLAB = 'google.colab' in sys.modules

if EN_COLAB:
    print("📍 Entorno: Google Colab")
    subprocess.run([
        sys.executable, "-m", "pip", "install",
        "ydata-profiling", "sweetviz", "-q"
    ])
else:
    print(f"📍 Entorno: Local — Python {sys.version.split()[0]}")
    # En local asumimos que el entorno ya tiene todo instalado.
    # Si falta algo: pip install ydata-profiling sweetviz

print("✅ Entorno listo.")

In [ ]:
# Instalacion de dependencias
# pip install -r requirements.txt

# Versiones de librerias 
!pip show pandas scikit-learn seaborn | grep -E "Name|Version"

# 📊 Análisis Exploratorio de Datos — Medical Insurance Costs
**Diplomado de Machine Learning · Sesión 1**

| | |
|---|---|
| **Dataset** | [Medical Insurance Costs](https://www.kaggle.com/datasets/mirichoi0218/insurance) |
| **Objetivo** | Explorar la estructura del dataset antes de construir un modelo predictivo |
| **Siguiente sesión** | Regresión lineal con este mismo dataset |

---
> Al finalizar esta sesión deberás poder describir la distribución de cada variable,
> identificar outliers y valores faltantes, y tener hipótesis sobre qué variables
> predicen mejor el costo del seguro.

In [ ]:
# ── Verificación de versiones ──────────────────────────────────────────────
import sys
print(f"Python      : {sys.version.split()[0]}")

import pandas as pd
import numpy as np
import matplotlib
import seaborn
import sklearn

print(f"pandas      : {pd.__version__}")
print(f"numpy       : {np.__version__}")
print(f"matplotlib  : {matplotlib.__version__}")
print(f"seaborn     : {seaborn.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

# ── Configuración global ───────────────────────────────────────────────────
warnings.filterwarnings("ignore")

# Constante de semilla para reproducibilidad
SEED = 42
np.random.seed(SEED)

# Estilo de gráficas
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"]  = 13
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False

# Opciones de display de pandas
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:.2f}".format)

print("✅ Configuración lista.")

In [ ]:
# ── Descarga de datos ─────────────────────────────────────────────────────
import subprocess
from pathlib import Path

print(f"Directorio actual del notebook: {Path.cwd()}")

# Buscar la raiz del proyecto que contiene la carpeta 01_Taller
repo_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "01_Taller").exists()),
    None
)
if repo_root is None:
    print("No se encontró la carpeta '01_Taller' desde el directorio actual.")
else:
    data_path = (repo_root / "01_Taller" / "data" / "insurance.csv").resolve()
    ruta_archivo = data_path
    print(f"Ruta esperada del dataset: {data_path}")
    if data_path.exists():
        print(f"Los datos se encuentran en: {data_path}")
    else:
        print("No se encontró el archivo. Descargando dataset con Kaggle CLI...")
        data_path.parent.mkdir(parents=True, exist_ok=True)

        try:
            subprocess.run(
                [
                    "kaggle", "datasets", "download",
                    "-d", "mirichoi0218/insurance",
                    "-p", str(data_path.parent),
                    "--unzip",
                ],
                check=True,
                cwd=repo_root
            )

            if data_path.exists():
                print(f"Descarga completada: {data_path}")
            else:
                print("La descarga terminó, pero no se encontró insurance.csv en la ruta esperada.")
        except FileNotFoundError:
            print("No se encontró el comando 'kaggle'. Instala Kaggle CLI y configura tu API token.")
        except subprocess.CalledProcessError as e:
            print(f"Error al descargar el dataset: {e}")


## 1. ¿Qué es el Análisis Exploratorio de Datos (EDA)?

El Análisis Exploratorio de Datos (EDA por sus siglas en inglés, Exploratory Data Analysis) es el proceso de examinar y comprender los conjuntos de datos para resumir sus características principales. A menudo, esto se hace utilizando métodos visuales y estadísticos.

Fue promovido originalmente por el estadístico estadounidense John Tukey en los años 70. Tukey argumentaba que antes de aplicar modelos estadísticos formales o algoritmos predictivos, es un paso crítico realizar un examen exhaustivo y sistemático de los datos empíricos para descubrir estructuras subyacentes, aislar variables importantes y detectar anomalías.

En términos prácticos con Python, el EDA consiste en:

* ***Conocer la forma de los datos***: ¿Cuántas filas y columnas tenemos?

* ***Identificar tipos de datos***: ¿Son números, textos, fechas? (Aquí repasaremos conceptos clave de variables en Python).

* ***Descubrir valores faltantes (nulos) o atípicos (outliers)***: Datos que se salen de la norma y pueden sesgar el análisis posterior.

* ***Encontrar patrones y correlaciones***: ¿Existe una relación medible entre diferentes variables, como el precio de una casa y su tamaño?

## 2. ¿Por qué es vital el EDA para el Machine Learning?

En Machine Learning, existe un principio fundamental conocido como GIGO: Garbage In, Garbage Out. Los algoritmos de Machine Learning son procesos matemáticos y estadísticos; si se alimentan con datos de mala calidad, optimizarán patrones incorrectos y generarán predicciones erróneas.

El EDA es el mecanismo preventivo estándar en la industria. Es crucial por estas razones:

* ***Limpieza y Calidad de Datos***: Los algoritmos no pueden procesar datos no estructurados o celdas vacías de forma nativa. El EDA nos permite detectar estos problemas para aplicar técnicas de imputación o transformación usando Python.

* ***Selección de Características (Feature Selection)***: No todas las variables de un conjunto de datos tienen el mismo poder predictivo. El EDA nos ayuda a identificar qué variables aportan información relevante, optimizando el tiempo de cómputo y el rendimiento del modelo.

* ***Detección de Desbalances y Sesgos (Bias)***: Nos permite evaluar la distribución de nuestras clases (por ejemplo, una proporción 90/10 en un problema de clasificación), previniendo que el modelo desarrolle un sesgo hacia la clase mayoritaria.

* ***Validación de Supuestos***: Muchos modelos de Machine Learning asumen distribuciones específicas en los datos de entrada. El EDA permite verificar estadística y visualmente el cumplimiento de dichas asunciones.

## 3. Tipos de Análisis en el EDA

Dependiendo de la cantidad de variables (columnas) que se examinen de manera simultánea, el EDA se clasifica en tres niveles principales:

1. ***Análisis Univariado***: Consiste en examinar una sola variable de forma aislada. El objetivo es describir los datos y encontrar la distribución y patrones dentro de esa única característica.
    * *Métricas clave*: Tendencia central (media, mediana, moda), dispersión (varianza, desviación estándar) y forma de la distribución.

2. ***Análisis Bivariado***: Implica el análisis simultáneo de dos variables para determinar las relaciones empíricas o correlaciones entre ellas.
    * *Métricas clave*: Coeficientes de correlación, covarianza y análisis de diferencias entre grupos.

3. ***Análisis Multivariado***: Examina más de dos variables simultáneamente. Es fundamental en conjuntos de datos complejos para entender interacciones conjuntas y dependencias estructurales.
    * Métricas clave: Matrices de correlación, análisis de componentes principales (PCA) o agrupaciones.

Refuerzo en Python: Nos permitirá introducir la creación de funciones personalizadas (def) para automatizar el cálculo de métricas sobre múltiples variables, aplicando el principio DRY (Don't Repeat Yourself)

In [ ]:
# --- Carga de datos ----------------------------------------------

df = pd.read_csv(ruta_archivo)
df_original = df.copy()   # copia de respaldo — nunca modificar esta

print(f"✅ Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")

## 4. Inspección inicial del conjunto de datos 

Antes de aplicar cualquier técnica estadística avanzada, debemos entender la anatomía de nuestro conjunto de datos. En Python, el estándar de la industria para la manipulación de datos tabulares es la librería `pandas`.

### Comandos básicos de inspección:
#### ¿Qué es exactamente la clase pandas.DataFrame?
Antes de avanzar, es fundamental definir qué es el objeto `df` que acabamos de crear. Un `DataFrame` es la estructura de datos bidimensional principal de la librería Pandas.

Puedes pensarlo como una hoja de cálculo en memoria o una tabla de una base de datos SQL: está compuesto por filas y columnas, y cada columna puede almacenar un tipo de dato distinto (números enteros, flotantes, cadenas de texto, booleanos). En el contexto de Python puro, un DataFrame actúa como un diccionario altamente optimizado, donde las claves son los nombres de las columnas y los valores son las listas de datos (que Pandas maneja como objetos llamados `Series`).

#### ¿Y qué es una pandas.Series?
Para comprender el funcionamiento interno de un DataFrame, debemos entender las `Series`. Si el DataFrame es la tabla completa, una `Series` es una única columna de esa tabla. Es una estructura de datos unidimensional (similar a una lista estándar de Python) pero mucho más robusta: posee un índice asociado (etiquetas para cada fila) y está optimizada para realizar operaciones vectorizadas (cálculos sobre todos sus elementos simultáneamente) a gran velocidad.

Una vez cargado nuestro `df`, ejecutaremos una serie de métodos estructurales. Observa cómo los resultados de Pandas se relacionan con los tipos de datos nativos de Python:

In [ ]:
# Vamos a verificar el tipo de dato de df 
print(type(df))

`df.head(n)` y `df.tail(n)`
*Muestra las primeras/últimas 'n' filas. Ideal para un vistazo rápido.*

**Nota:** `display()` es una función de IPython que renderiza objetos  como tablas HTML en el notebook. Fuera de Colab/Jupyter, usa `print()` en su lugar.

In [ ]:
print("\nPrimeras 5 filas del dataset:")
display(df.head(5))

In [ ]:
print("\nÚltimas 5 filas del dataset:")
display(df.tail(5))

`df.shape`
*Devuelve (filas, columnas). Es una 'tupla' inmutable nativa de Python.*

In [ ]:
dimensiones = df.shape
print(dimensiones)
print(f"\nDimensiones del dataset: {dimensiones[0]} filas x {dimensiones[1]} columnas")

`df.columns` y `df.index` *Nombres de las columnas y etiquetas de las filas. Se pueden convertir a 'listas'.*

In [ ]:
columnas_lista = list(df.columns)
print(f"\nLista de columnas: {columnas_lista}")

`df.info()` *Resumen técnico rápido: conteo de nulos, tipos de datos y memoria usada.*

**💡 ¿Qué observar en df.info()?**

- **Non-Null Count** igual al total de filas → no hay nulos en esa columna
- **Dtype `object`** → variable categórica o de texto (candidata a encoding)
- **Dtype `float64` vs `int64`** → ¿tiene sentido para esa variable?

En nuestro dataset: ¿hay columnas que deberían ser numéricas pero aparecen como `object`?

In [ ]:
print("\n--- Información General (df.info) ---")
df.info()

`df.dtypes` *Muestra solo el tipo de dato específico de cada columna (int, float, object).*

In [ ]:
print("\n--- Tipos de Datos (df.dtypes) ---")
print(df.dtypes)

#### 💡 ¿`str` u `object`? Depende de la versión de Pandas

En **Pandas < 2.0**, las columnas de texto siempre aparecían como `object`.
En **Pandas ≥ 2.0**, pueden aparecer como `str` (StringDtype nativo).

Funcionalmente son equivalentes para nuestro análisis, pero si ves comportamientos inesperados al aplicar `.str.lower()` o `pd.get_dummies()`, puedes forzar el tipo explícitamente:

```python
df['smoker'] = df['smoker'].astype('category')
```

Usar `category` es más eficiente en memoria para variables con pocos valores únicos.

In [ ]:
df['sex'] = df['sex'].astype('category')

`df.describe()` *Estadísticas matemáticas básicas (media, min, max, cuartiles) solo para números.*

In [ ]:
print("\n--- Resumen Estadístico (df.describe) ---")
display(df.describe())

In [ ]:
print("\n--- Tipos de Datos (df.dtypes) ---")
print(df.dtypes)

`df.describe(include=['object', 'bool'])` *Estadísticas para variables no numéricas (categorías de texto y booleanos).*

In [ ]:
print("\n--- Resumen Estadístico de Variables Categóricas ---")
display(df.describe(include=['category', 'object', 'bool']))

`df.describe(include='all')` *Muestra estadísticas de TODAS las columnas (numéricas y categóricas juntas).*

**¿Por qué aquí usamos un string ('all') y arriba una lista ['object', 'bool']?**
- Usamos una LISTA cuando queremos filtrar indicando varios tipos de datos específicos de Python/Pandas.
- Usamos el STRING 'all' porque es una palabra clave (keyword) especial reservada por Pandas que funciona como un atajo para decirle a la función: "ignora los tipos de datos y tráelo todo".

In [ ]:
print("\n--- Resumen Estadístico Total ---")
display(df.describe(include='all'))

#### 💡 ¿Por qué hay NaN en df.describe(include='all')?

El `NaN` en esta tabla **no significa datos faltantes** — significa que esa estadística no aplica para ese tipo de variable:

- `mean`, `std`, `min`, `max` → solo tienen sentido para variables **numéricas**
- `unique`, `top`, `freq` → solo tienen sentido para variables **categóricas**

Por eso `describe(include='all')` es útil para tener todo en una vista,
pero `describe()` sola y `describe(include='object')` sola dan tablas más limpias. Como se mostró en el ejemplo anterior.

In [ ]:
# Antes de avanzar vamos a cmabiar las variables categóricas a tipo 'category' para optimizar memoria y análisis posterior
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df[col] = df[col].astype('category')

print("\n--- Tipos de Datos (df.dtypes) ---")
print(df.dtypes)

### ✅ Lo que sabemos hasta aquí

| Variable | Tipo | Observación inicial |
|----------|------|---------------------|
| `age` | int64 | Rango 18–64, media 39.2, distribución por explorar |
| `bmi` | float64 | Media 30.7 — justo en el límite de obesidad (≥30) |
| `charges` | float64 | Media $13,270 pero std $12,110 — alta variabilidad, probable sesgo |
| `children` | int64 | 75% tiene 2 hijos o menos, máximo 5 |
| `smoker` | str/category | 1,064 de 1,338 son no fumadores — dataset desbalanceado |
| `sex` | str/category | 676 male vs 662 female — prácticamente balanceado |
| `region` | str/category | Southeast es la región más frecuente (364 registros) |

**Preguntas abiertas para la siguiente sección:**
- ¿La alta std de `charges` se explica por los fumadores?
- ¿El bmi tiene outliers por encima de 50?
- ¿Hay nulos en alguna columna? → `df.info()` no mostró ninguno, pero lo confirmaremos

## 5. Análisis Univariado

El Análisis Univariado examina **una sola variable a la vez**, aislándola del resto. El objetivo no es encontrar relaciones entre variables, sino descubrir la estructura interna, la calidad y los patrones propios de cada característica.

Dependiendo del tipo de variable, el enfoque es distinto:

| Tipo | ¿Qué analizamos? | Visualizaciones clave |
|------|------------------|-----------------------|
| **Categórica** | Frecuencias, proporciones, desbalances | Barplot, pie chart |
| **Numérica** | Distribución, tendencia central, dispersión, forma | Histograma, KDE, boxplot |


### Variables Categóricas (Cualitativas)

Representan cualidades, grupos o estados (ej. Género, Categoría de cliente, Activo/Inactivo). No tienen un sentido matemático directo (no podemos calcular el "promedio" del género), por lo que nos enfocamos en el análisis de frecuencias.

* ***Frecuencia Absoluta***: ¿Cuántos registros exactos pertenecen a cada categoría? (Ej. 40 clientes en la Categoría A).

* ***Frecuencia Relativa (Proporción)***: ¿Qué porcentaje del total representa cada categoría? Esto es vital para descubrir desbalances en los datos, un problema gravísimo en Machine Learning.

* ***Visualizaciones clave***: Gráficos de barras (Bar charts), gráficos de pastel (Pie charts).

### Variables Numéricas (Cuantitativas)

Representan magnitudes medibles (ej. Edad, Salario, Temperatura). Aquí aplicamos todo el peso de la estadística descriptiva para entender su distribución:

* ***Medidas de Tendencia Central***: ¿Dónde se agrupan o concentran los datos?

* ***Media (Promedio)***: El centro de gravedad, aunque extremadamente sensible a valores anómalos (outliers).

* ***Mediana***: El valor que divide los datos exactamente por la mitad (percentil 50). Es nuestro mejor aliado cuando los datos están sesgados.

* ***Moda***: El valor que más se repite frecuentemente.

* ***Medidas de Dispersión***: ¿Qué tan "estirados" o variados están los datos respecto a su centro?

* ***Desviación Estándar y Varianza***: Indican el grado de variación o "ruido" promedio de la variable.

* ***Rango***: La distancia total matemática entre el valor máximo y el mínimo.

* ***Rango Intercuartílico (IQR)***: La distancia entre el percentil 25 (Q1) y el percentil 75 (Q3). Es la técnica estándar de la industria para aislar estadísticamente los outliers.

#### Medidas de Forma:

* ***Asimetría (Skewness)***: Determina si la campana de datos se inclina hacia la izquierda o la derecha.

* ***Curtosis***: Mide qué tan puntiaguda (datos muy concentrados) o aplastada (datos muy dispersos) es la distribución.

* ***Visualizaciones clave***: Histogramas, gráficos de densidad (KDE), diagramas de caja y bigotes (Boxplots).

### Análisis de variables categoricas 

Para entender verdaderamente qué hace Pandas por detrás, vamos a resolver el problema de contar categorías usando las bases de Python.

In [ ]:
# Conteo usando Diccionarios, Listas y Bucles For
print("--- 1. Forma Manual (Python puro) ---")
# 1. Creamos un diccionario vacío para almacenar nuestro conteo
conteo_manual = {} 

# 2. Extraemos los datos a una lista de Python (ignorando nulos para no romper el código)
lista_categorias = df['sex'].dropna().tolist()

# 3. Iteramos sobre la lista con un bucle 'for'
for categoria in lista_categorias:
    # Condicional if/else: Si la categoría ya existe como clave en el diccionario, sumamos 1
    if categoria in conteo_manual:
        conteo_manual[categoria] += 1
    # Si no existe, la creamos y le asignamos el valor inicial de 1
    else:
        conteo_manual[categoria] = 1

print(f"Diccionario resultante: {conteo_manual}")

print("\n--- 2. Forma Optimizada (Librería Pandas) ---")
# En el mundo real usamos Pandas, que ejecuta este mismo proceso en código C ultra-rápido por debajo.
conteo_pandas = df['sex'].value_counts()
print(type(conteo_pandas))
print(f"Conteo para 'male': {conteo_pandas.loc['male']}")
print(f"Conteo para 'female': {conteo_pandas.loc['female']}")
conteo_dict = conteo_pandas.to_dict()
print(f"Diccionario resultante: {conteo_dict}")

In [ ]:
print("--- Análisis Automático de Variables Categóricas ---")

# Iteramos sobre todos los nombres de las columnas
for nombre_col in df.columns:
    # Comprobamos si el tipo de dato es 'category' o 'bool'
    if df[nombre_col].dtype == 'category' or df[nombre_col].dtype == 'bool':
        print(f"\n>> Variable Categórica detectada: '{nombre_col}'")
        display(df[nombre_col].value_counts())
        print("-" * 40)

In [ ]:
# ── Análisis automático de todas las variables categóricas ─────────────────
# pd.api.types es robusto para Pandas < 2.0 y >= 2.0
print("=== Análisis Automático — Variables Categóricas ===\n")

for nombre_col in df.columns:
    es_categorica = (
        pd.api.types.is_string_dtype(df[nombre_col]) or
        pd.api.types.is_bool_dtype(df[nombre_col]) or
        pd.api.types.is_categorical_dtype(df[nombre_col])
    )

    if es_categorica:
        n_unicos = df[nombre_col].nunique()
        print(f"{'─'*45}")
        print(f"  Variable : {nombre_col}")
        print(f"  Categorías únicas ({n_unicos}): {df[nombre_col].unique().tolist()}")
        print()

        # Tabla de frecuencia absoluta y relativa
        tabla = pd.DataFrame({
            "Frecuencia absoluta" : df[nombre_col].value_counts(),
            "Frecuencia relativa (%)" : df[nombre_col].value_counts(normalize=True)
                                          .mul(100).round(1)
        })
        display(tabla)
        print()

### Visualización de Variables Categóricas (Visualización Base)

Las tablas de texto a menudo no son suficientes; necesitamos gráficos. Para visualizarlas, introduciremos las dos librerías más famosas de visualización en Python: matplotlib y seaborn.

In [ ]:
col = 'region'
orden = df[col].value_counts().index

fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(
        data=df, x=col, order=orden, ax=ax,
        palette="muted", edgecolor="white"
)

ax.set_title(f"Distribución de {col}")
ax.set_xlabel("")
ax.set_ylabel("Frecuencia")

total = len(df)
for p in ax.patches:
        pct = f"{100 * p.get_height() / total:.1f}%"
        ax.annotate(
                pct,
                (p.get_x() + p.get_width() / 2, p.get_height() + 5),
                ha="center", va="bottom", fontsize=10
        )

plt.tight_layout()
plt.show()


In [ ]:
# ── Barplots para variables categóricas ────────────────────────────────────
categoricas = ['sex', 'smoker', 'region']
fig, axes = plt.subplots(3, 1, figsize=(10, 12))
axes = axes.flatten()  # <- convierte (2,2) en lista de ejes

for ax, col in zip(axes, categoricas):
    orden = df[col].value_counts().index
    sns.countplot(
        data=df, x=col, order=orden, ax=ax,
        palette="muted", edgecolor="white"
    )
    ax.set_title(f"Distribución de '{col}'")
    ax.set_xlabel("")
    ax.set_ylabel("Frecuencia")

    total = len(df)
    for p in ax.patches:
        pct = f"{100 * p.get_height() / total:.1f}%"
        ax.annotate(
            pct,
            (p.get_x() + p.get_width() / 2, p.get_height() + 5),
            ha='center', va='bottom', fontsize=10
        )

# Ocultar ejes no usados (hay 4 subplots y solo 3 variables)
for ax in axes[len(categoricas):]:
    ax.set_visible(False)

plt.suptitle("Variables Categóricas — Frecuencias", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
col = 'smoker'
conteo = df[col].value_counts()

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    conteo.values,
    labels=conteo.index,
    autopct='%1.1f%%',
    startangle=90,
    counterclock=False,
    wedgeprops={"edgecolor": "white"}
)

ax.set_title(f"Distribución de {col}")
ax.axis("equal")  # mantiene el gráfico circular
plt.tight_layout()
plt.show()


In [ ]:
# ── Piecharts para variables categóricas (2 columnas) ──────────────────────
categoricas = ['sex', 'smoker', 'region']

n_cols = 1
n_rows = int(np.ceil(len(categoricas) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 4.5 * n_rows))
axes = np.array(axes).reshape(-1)  # asegura vector de ejes

for ax, col in zip(axes, categoricas):
    conteo = df[col].value_counts()
    ax.pie(
        conteo.values,
        labels=conteo.index,
        autopct='%1.1f%%',
        startangle=90,
        counterclock=False,
        wedgeprops={"edgecolor": "white"}
    )
    ax.set_title(f"Distribución de '{col}'")
    ax.axis("equal")

# Ocultar ejes no usados
for ax in axes[len(categoricas):]:
    ax.set_visible(False)

plt.suptitle("Variables Categóricas — Proporciones", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


#### 5.2 Variables Numéricas

Para las variables numéricas, no basta con la media. Necesitamos conocer su dispersión y su "forma". En ML, evaluamos los cuatro momentos estadísticos:

* ***Media***: Centro de masa.

* ***Desviación Estándar***: Dispersión promedio.

* ***Asimetría (Skewness)***: Mide el sesgo. Modelos de Regresión Lineal asumen normalidad; si la asimetría es muy alta ($> 1$), la regresión será atraída hacia los valores extremos.

* ***Curtosis (Kurtosis)***: Mide el peso de las colas. Alta curtosis indica alta probabilidad de outliers severos.

| Dimensión | Métricas | ¿Qué revela? |
|-----------|----------|--------------|
| **Tendencia central** | Media, mediana, moda | Dónde se concentran los datos |
| **Dispersión** | Std, varianza, rango, IQR | Qué tan "estirados" están |
| **Forma** | Skewness, curtosis | Si la distribución es simétrica y qué tan pesadas son sus colas |

> **Regla práctica para skewness:**
> - \|skewness\| < 0.5 → distribución aproximadamente simétrica ✅
> - 0.5 < \|skewness\| < 1 → sesgo moderado ⚠️
> - \|skewness\| > 1 → sesgo fuerte, considerar transformación logarítmica 🔴

In [ ]:
# ── Estadísticas descriptivas para variables numéricas ─────────────────────
numericas = df.select_dtypes(include='number').columns.tolist()
print(f"Variables numéricas detectadas: {numericas}\n")

# Tabla extendida: describe() + skewness + curtosis
resumen = df[numericas].describe().T
resumen['skewness']  = df[numericas].skew()
resumen['curtosis']  = df[numericas].kurtosis()
resumen['IQR']       = df[numericas].quantile(0.75) - df[numericas].quantile(0.25)

display(resumen.style.format("{:.2f}").background_gradient(
    subset=['skewness'], cmap='RdYlGn_r'))

#### Validaciones Visuales (KDE)

Los números anteriores son un diagnóstico matemático. Vamos a comprobarlo mediante densidades suavizadas (KDE - Kernel Density Estimation).

In [ ]:
# ── Histograma y boxplot para cada variable numérica ──────────────────────
# Histograma + KDE (forma) y Boxplot (outliers) siempre juntos:
# el primero muestra la distribución, el segundo revela valores extremos.

fig, axes = plt.subplots(len(numericas), 2,
                         figsize=(12, 4 * len(numericas)))

for i, col in enumerate(numericas):
    # Histograma + KDE
    sns.histplot(data=df, x=col, kde=True,
                 ax=axes[i, 0], color='steelblue', edgecolor='white')
    axes[i, 0].set_title(f"{col} — Histograma + KDE")
    axes[i, 0].axvline(df[col].mean(),   color='red',    ls='--',
                       lw=1.2, label=f"Media: {df[col].mean():.1f}")
    axes[i, 0].axvline(df[col].median(), color='green',  ls='--',
                       lw=1.2, label=f"Mediana: {df[col].median():.1f}")
    axes[i, 0].legend(fontsize=9)

    # Boxplot
    sns.boxplot(data=df, y=col, ax=axes[i, 1],
                color='steelblue', width=0.4)
    axes[i, 1].set_title(f"{col} — Boxplot")

plt.suptitle("Variables Numéricas — Distribución", fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

#### 💡 Cómo leer estos gráficos juntos

- Si **media > mediana** (línea roja a la derecha de la verde) → sesgo positivo
- Si las líneas son casi iguales → distribución simétrica
- En el **boxplot**, los puntos fuera de los bigotes son outliers candidatos

**Lo que vemos en nuestro dataset:**
- `charges`: media muy por encima de la mediana + cola larga a la derecha
  → sesgo positivo fuerte, confirmado ✅
- `bmi`: media ≈ mediana, forma casi normal con algunos outliers por arriba
- `age`: distribución inusualmente plana (casi uniforme) — no es una campana,
  los pacientes están distribuidos de forma pareja entre 18 y 64 años

### ✅ Síntesis del Análisis Univariado

| Variable | Tipo | Hallazgo clave | Acción para el modelo |
|----------|------|----------------|----------------------|
| `age` | Numérica | Distribución uniforme, sin outliers | Usar directamente |
| `bmi` | Numérica | Casi normal, outliers leves por arriba | Revisar con IQR |
| `charges` | Numérica | Sesgo fuerte positivo | Evaluar `log1p` |
| `children` | Numérica | Discreta, sesgo moderado | Usar directamente |
| `sex` | Categórica | Balanceada (50/50) | Encoding binario |
| `smoker` | Categórica | Desbalanceada (79/21) ⚠️ | Encoding binario, alta relevancia esperada |
| `region` | Categórica | 4 categorías equilibradas | One-hot encoding (3 dummies) |

**Hipótesis hacia el análisis bivariado:**
- `smoker` probablemente explica gran parte de la varianza en `charges`
- `bmi` y `age` tendrán correlación positiva con `charges`
- `region` y `sex` podrían tener bajo poder predictivo individual

→ Lo verificamos en la **Sección 6: Análisis Bivariado**

### 🔧 Tu turno

1. La variable `charges` tiene skewness fuerte. Aplica `np.log1p()` y
   grafica el histograma de la versión transformada. ¿Mejora la simetría?

2. ¿Cuál es el IQR de `bmi`? ¿Cuántos registros caen fuera del rango
   `[Q1 - 1.5×IQR, Q3 + 1.5×IQR]`?

3. Agrega una columna `obeso` al dataframe que sea `True` si `bmi >= 30`
   y `False` en caso contrario. ¿Qué porcentaje del dataset es obeso?

In [ ]:
# Pon tú código aquí

## 6. Análisis Bivariado

En el análisis univariado describimos cada variable por separado.
Ahora cruzamos variables para buscar **relaciones, patrones y dependencias**.

La estrategia depende de los tipos de variables que cruzamos:

| Combinación | Pregunta que responde | Herramientas |
|-------------|----------------------|--------------|
| **Num × Num** | ¿Se mueven juntas? ¿En qué dirección? | Correlación, scatterplot, pairplot |
| **Num × Cat** | ¿Cambia la distribución numérica según el grupo? | Boxplot, violinplot, t-test |
| **Cat × Cat** | ¿Hay asociación entre dos grupos? | Tabla de contingencia, chi² |

> **Hipótesis que traemos del análisis univariado:**
> - `smoker` explica gran parte de la varianza en `charges`
> - `bmi` y `age` tienen correlación positiva con `charges`
> - `region` y `sex` tienen bajo poder predictivo individual

### Análisis Bivariado y Multivariado: Interacciones Numéricas (Espacio $R^n$)

Una vez comprendida la distribución marginal de cada variable (Análisis Univariado), el siguiente paso analítico es estudiar la distribución conjunta en el espacio numérico continuo (Num $\times$ Num). En Machine Learning, buscamos dos propiedades algebraicas fundamentales al cruzar estas variables:

* ***Poder Predictivo (Feature vs Target)***: ¿Existe una correlación fuerte entre nuestro vector de características ($X$) y el vector objetivo ($y$)? Esto define el límite superior de rendimiento de nuestros modelos simples.

* ***Ausencia de Multicolinealidad (Feature vs Feature)***: Esto es vital. Los estimadores de la Regresión Lineal Ordinaria (OLS) se calculan matricialmente como $\hat{\beta} = (X^T X)^{-1} X^T y$. Si dos características están altamente correlacionadas entre sí, las columnas de la matriz de diseño $X$ dejan de ser linealmente independientes. Como consecuencia, el determinante de $(X^T X)$ tiende a cero, volviendo a la matriz casi singular (mal condicionada). Al invertir una matriz casi singular, la varianza de los coeficientes $\hat{\beta}$ explota, volviendo al modelo completamente inestable y dependiente del ruido.


### Matriz de Covarianza y Correlación No Paramétrica (Spearman)

La inmensa mayoría de los analistas utilizan la correlación de Pearson por defecto (df.corr()). Matemáticamente, Pearson asume que la relación es estrictamente lineal y que las variables provienen de una distribución normal multivariada.

Como ya demostramos en el Paso 2 que charges tiene un sesgo extremo (Skew = $1.52$) y colas pesadas, aplicar Pearson está matemáticamente contraindicado; subestimará drásticamente la relación real ya que los coeficientes se verán fuertemente arrastrados por los outliers.

Para mantener la rigurosidad estadística, utilizaremos el Coeficiente de Correlación de Spearman ($\rho$). Este método aplica una transformación topológica: convierte los valores continuos a sus respectivos rangos ordinales (posiciones relativas) y evalúa la relación sobre esos rangos, capturando así relaciones monotónicas puras y siendo extremadamente robusto ante la no-normalidad de los datos médicos.


### Numérica × Numérica — Correlación

La **correlación** mide la fuerza y dirección de la relación lineal entre dos variables numéricas.
El coeficiente de Pearson (*r*) varía entre -1 y 1:

| Valor de *r* | Interpretación |
|---|---|
| 0.9 a 1.0 | Correlación positiva muy fuerte |
| 0.5 a 0.9 | Correlación positiva moderada a fuerte |
| 0.1 a 0.5 | Correlación positiva débil |
| ~0.0 | Sin correlación lineal |
| Negativo | Correlación inversa |

> **Limitación importante:** Pearson solo detecta relaciones **lineales**.
> Usa Spearman cuando haya outliers fuertes o relaciones no lineales.
> Recuerda: `charges` tiene skewness de **1.52** — Spearman será más robusto aquí.

In [ ]:
# ── Correlación de Pearson y Spearman ──────────────────────────────────────
numericas = df.select_dtypes(include='number').columns.tolist()

corr_pearson  = df[numericas].corr(method='pearson')
corr_spearman = df[numericas].corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, corr, titulo in zip(
    axes,
    [corr_pearson, corr_spearman],
    ["Correlación de Pearson", "Correlación de Spearman"]
):
    sns.heatmap(
        corr, annot=True, fmt=".2f", cmap="coolwarm",
        vmin=-1, vmax=1, center=0,
        square=True, linewidths=0.5, ax=ax,
        annot_kws={"size": 11}
    )
    ax.set_title(titulo, fontsize=13, fontweight='bold')

plt.suptitle("Matrices de Correlación — Variables Numéricas",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Tabla comparativa enfocada en charges
print("\n── Correlación con 'charges' ────────────────────────────────")
comparativa = pd.DataFrame({
    "Pearson" : corr_pearson['charges'].drop('charges'),
    "Spearman": corr_spearman['charges'].drop('charges')
}).sort_values('Spearman', ascending=False)

display(comparativa.style.format("{:.3f}").background_gradient(
    cmap='RdYlGn', vmin=-1, vmax=1))

**💡 ¿Qué buscar en el heatmap?**

Con base en nuestros datos:

| Par de variables | Pearson esperado | Spearman esperado | Interpretación |
|---|---|---|---|
| `age` × `charges` | ~0.30 | ~0.53 | Spearman mayor → relación no lineal, con outliers |
| `bmi` × `charges` | ~0.20 | ~0.41 | Mismo patrón — outliers distorsionan Pearson |
| `children` × `charges` | ~0.07 | ~0.07 | Sin correlación relevante |
| `age` × `bmi` | ~0.11 | ~0.11 | Prácticamente independientes |

> Cuando Spearman > Pearson de forma consistente, es señal de que
> los outliers o el sesgo de `charges` (skewness 1.52, IQR $11,899$)
> están atenuando artificialmente la correlación lineal.
> Esto refuerza la idea de aplicar `log1p` a `charges` antes de modelar.

### Proyección Topológica Bidimensional (Pairplot)

Como nos advirtió la estadística clásica a través del famoso Cuarteto de Anscombe, depender exclusivamente de matrices de covarianza escalar es peligroso, ya que colapsan distribuciones bidimensionales complejas a un simple número. Es imperativo proyectar el subespacio continuo de datos topológicamente.

Crearemos una cuadrícula de dispersión $R^2$, pero introduciremos un enfoque analítico avanzado: proyectaremos una tercera dimensión categórica discreta (smoker) mediante un mapa de color (hue). Esto revelará si un subgrupo poblacional obedece a una función matemática generadora de datos distinta.

In [ ]:
# ── Scatterplots: age y bmi vs charges ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, var in zip(axes, ['age', 'bmi']):
    sns.scatterplot(
        data=df, x=var, y='charges',
        hue='smoker', palette={'yes': '#E63946', 'no': '#457B9D'},
        alpha=0.6, edgecolor='white', linewidth=0.3, ax=ax
    )
    ax.set_title(f"'{var}' vs 'charges'  (color = smoker)")
    ax.set_xlabel(var)
    ax.set_ylabel("charges ($)")

plt.suptitle("Relaciones Numéricas con charges",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**💡 Lo que revelan los scatterplots**

Coloreando por `smoker` aparece algo que la correlación numérica no capturaba:

- Los datos **no forman una nube continua** — forman **dos bandas paralelas**:
  una baja (no fumadores) y una alta (fumadores).
- Dentro de cada banda sí existe una tendencia lineal positiva con `age` y `bmi`.
- Esto explica por qué la correlación global `age × charges` es moderada (~0.30):
  la variable `smoker` actúa como un **efecto de confusión** que parte la nube en dos.

> **Implicación para regresión lineal:** el modelo necesita `smoker` como variable
> independiente para capturar este efecto. Sin ella, los coeficientes de `age`
> y `bmi` quedarán sesgados.

In [ ]:
# ── Pairplot general ───────────────────────────────────────────────────────
# Muestra todas las combinaciones numéricas + distribuciones en la diagonal.
# hue='smoker' revela el efecto de confusión en todas las relaciones a la vez.

g = sns.pairplot(
    df, hue='smoker',
    palette={'yes': '#E63946', 'no': '#457B9D'},
    plot_kws={'alpha': 0.4, 'edgecolor': 'none'},
    diag_kind='kde',
    corner=True          # muestra solo el triángulo inferior, más limpio
)
g.figure.suptitle("Pairplot — Variables Numéricas  (color = smoker)",
                  y=1.02, fontsize=13, fontweight='bold')
plt.show()

### Numérica × Categórica

Queremos saber: **¿cambia la distribución de `charges` según el grupo categórico?**

Herramientas:
- **Boxplot agrupado:** compara medianas y rangos IQR entre grupos
- **Violinplot:** muestra además la forma completa de la distribución
- **t-test (dos grupos) / ANOVA (tres o más):** contrasta si la diferencia
  de medias es estadísticamente significativa o podría ser ruido

Recordar: el IQR de `charges` es **$11,899$** y la mediana es **$9,382$**.
Cualquier diferencia de medianas entre grupos que supere ~$3,000$
merece atención.

In [ ]:
# ── charges según cada variable categórica ────────────────────────────────
categoricas = ['smoker', 'sex', 'region']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, categoricas):
    orden = (df.groupby(col)['charges']
               .median()
               .sort_values(ascending=False)
               .index)

    sns.boxplot(data=df, x=col, y='charges', order=orden,
                palette='muted', width=0.5, ax=ax,
                flierprops=dict(marker='o', markersize=3,
                                alpha=0.4, color='gray'))

    # Añadir mediana como anotación
    for i, cat in enumerate(orden):
        mediana = df.loc[df[col] == cat, 'charges'].median()
        ax.text(i, mediana + 400, f"${mediana:,.0f}",
                ha='center', va='bottom', fontsize=9, color='black')

    ax.set_title(f"charges por '{col}'")
    ax.set_xlabel("")
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))

plt.suptitle("Charges según Variables Categóricas",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Violinplot: charges por smoker ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

sns.violinplot(data=df, x='smoker', y='charges',
               palette={'yes': '#E63946', 'no': '#457B9D'},
               inner='quartile', ax=ax)

ax.set_title("Distribución de charges por smoker", fontsize=13, fontweight='bold')
ax.set_xlabel("Fumador")
ax.set_ylabel("Charges ($)")
ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))
plt.show()

# ── Prueba t de Student: ¿la diferencia de medias es significativa? ────────
from scipy import stats

grupo_si  = df.loc[df['smoker'] == 'yes',  'charges']
grupo_no  = df.loc[df['smoker'] == 'no',   'charges']

t_stat, p_valor = stats.ttest_ind(grupo_si, grupo_no, equal_var=False)

print(f"Media fumadores    : ${grupo_si.mean():>10,.2f}")
print(f"Media no fumadores : ${grupo_no.mean():>10,.2f}")
print(f"Diferencia         : ${grupo_si.mean() - grupo_no.mean():>10,.2f}")
print(f"\nt-statistic : {t_stat:.4f}")
print(f"p-valor     : {p_valor:.2e}")
print(f"\n{'✅ Diferencia estadísticamente significativa (p < 0.05)' if p_valor < 0.05 else '⚠️ No hay evidencia suficiente de diferencia'}")

### 💡 Hallazgos — charges por variable categórica

| Variable | Diferencia de medianas | ¿Significativa? | Conclusión |
|---|---|---|---|
| `smoker` | ~$23,000 entre grupos | ✅ Sí (p << 0.05) | **Predictor dominante** |
| `sex` | < $1,000 | ⚠️ Marginal | Poco poder predictivo individual |
| `region` | < $2,000 | ⚠️ Marginal | Efecto pequeño |

El violinplot de `smoker` revela además que:
- Los **no fumadores** tienen una distribución unimodal concentrada en valores bajos
- Los **fumadores** tienen una distribución **bimodal** — hay dos subgrupos dentro
  de los fumadores (probablemente relacionado con el nivel de `bmi`)

> Esta bimodalidad dentro de fumadores sugiere una **interacción** entre
> `smoker` y `bmi` — otro término a considerar en el modelo de regresión.

### Categórica × Categórica

En este bloque, el objetivo es determinar si la pertenencia a una clase categórica (ej. ser hombre o mujer) influye en la probabilidad de pertenecer a otra clase (ej. ser fumador o no). Si las variables fueran independientes, la probabilidad conjunta $P(A \cap B)$ debería ser igual al producto de sus probabilidades marginales $P(A) \cdot P(B)$.

**Test de Independencia Chi-Cuadrado ($\chi^2$) de Pearson**

Para validar si la relación observada es estadísticamente significativa, aplicamos una prueba de hipótesis no paramétrica.

1. Definición de Hipótesis:

    * $H_0$ (Hipótesis Nula): Las variables son independientes (No existe asociación).

    * $H_1$ (Hipótesis Alternativa): Las variables son dependientes (Existe una asociación estructural).

2. El Estadístico $\chi^2$:
Se calcula comparando las frecuencias observadas ($O$) contra las frecuencias esperadas ($E$) bajo el supuesto de independencia:

$$\chi^2 = \sum \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$

Donde $$E_{ij} = \frac{(\text{Total Fila}_i) \cdot (\text{Total Columna}_j)}{\text{Gran Total}}$$

3. Grados de Libertad ($df$):


$$df = (r - 1) \cdot (c - 1)$$

Donde $r$ y $c$ son el número de filas y columnas respectivamente.

Para cruzar dos variables categóricas usamos:
- **Tabla de contingencia:** frecuencia conjunta de cada combinación de categorías
- **Prueba chi²:** contrasta si las dos variables son independientes entre sí
  - H₀: las variables son independientes (no hay asociación)
  - H₁: existe asociación entre las variables
  - Si p-valor < 0.05 → rechazamos H₀ → hay asociación estadística

In [ ]:
# ── Tablas de contingencia ─────────────────────────────────────────────────
from scipy.stats import chi2_contingency

pares = [('smoker', 'sex'), ('smoker', 'region'), ('sex', 'region')]

for col1, col2 in pares:
    print(f"\n{'═'*50}")
    print(f"  {col1.upper()} × {col2.upper()}")
    print(f"{'═'*50}")

    tabla = pd.crosstab(df[col1], df[col2],
                        margins=True, margins_name="Total")
    display(tabla)

    # Chi² sobre la tabla sin márgenes
    tabla_sin_margenes = pd.crosstab(df[col1], df[col2])
    chi2, p, dof, esperados = chi2_contingency(tabla_sin_margenes)

    print(f"\n  chi² = {chi2:.3f}  |  gl = {dof}  |  p-valor = {p:.4f}")
    print(f"  → {'Asociación significativa ✅' if p < 0.05 else 'Sin asociación significativa ⚠️'}")

### ✅ Síntesis del Análisis Bivariado

| Relación | Hallazgo | Implicación para el modelo |
|---|---|---|
| `age` × `charges` | Correlación positiva moderada, no lineal | Incluir `age`; evaluar término cuadrático |
| `bmi` × `charges` | Correlación positiva, amplificada por `smoker` | Incluir `bmi`; explorar interacción con `smoker` |
| `children` × `charges` | Sin correlación relevante | Incluir con baja expectativa de coeficiente |
| `smoker` × `charges` | Diferencia de ~$23,000 en medias, p << 0.05 | **Variable más importante del modelo** |
| `sex` × `charges` | Sin diferencia significativa | Incluir pero no esperar peso alto |
| `region` × `charges` | Diferencia pequeña | Incluir para no omitir varianza explicable |
| `smoker` × `sex` | Sin asociación (chi²) | No hay sesgo de selección entre grupos |

**Ecuación candidata para regresión lineal (próxima sesión):**
```
charges = β₀ + β₁·age + β₂·bmi + β₃·children + β₄·smoker + β₅·sex + β₆·region + ε
```

La interacción `smoker × bmi` queda como variable de exploración avanzada.

### 🔧 Tu turno

1. Aplica `np.log1p()` a `charges` y recalcula la correlación de Pearson
   con `age` y `bmi`. ¿Mejoran los coeficientes respecto a los originales?

2. Crea un boxplot de `bmi` por `smoker`. ¿Los fumadores tienen mayor BMI?
   ¿Esto apoya la hipótesis de interacción `smoker × bmi`?

3. Usa ANOVA (`scipy.stats.f_oneway`) para contrastar si las medias de
   `charges` son iguales entre las 4 regiones. ¿Qué concluyes?

In [ ]:
# Pon tú código aquí


## 7. Detección de Anomalías y Calidad del Dato

En esta fase, auditamos la "salud" de nuestros datos. Las anomalías pueden ser de dos tipos:

* ***Anomalías Lógicas***: Datos que violan las leyes del dominio (ej. edad negativa, BMI de cero).

* ***Anomalías Estadísticas (Outliers)***: Datos que son matemáticamente distantes del resto de la muestra. En ML, los outliers en el target (charges) sesgan la media y los residuos, mientras que en las features (bmi) pueden tener un alto apalancamiento (leverage) y deformar la frontera de decisión.



| Pregunta | Herramienta |
|---|---|
| ¿Hay registros duplicados? | `duplicated()` |
| ¿Hay valores fuera de rango lógico? | Inspección de mínimos y máximos |
| ¿Hay outliers estadísticos? | IQR · Z-score |

> **¿Por qué importa esto para ML?**
> Un outlier no tratado puede desplazar los coeficientes de regresión lineal
> de forma severa — el método de mínimos cuadrados minimiza errores al
> **cuadrado**, lo que amplifica el efecto de valores extremos.

#### Duplicados
Un registro duplicado introduce sesgo: el modelo aprende ese patrón
con más peso del que merece. En datasets pequeños como el nuestro
(1,338 filas), un solo duplicado representa el 0.07% del total —
parece poco, pero en producción puede indicar un problema sistemático
de ingesta de datos.

#### Rangos inválidos
Cada variable tiene un rango lógico conocido. Valores fuera de ese
rango no son outliers estadísticos — son **errores de datos**:

| Variable | Rango válido esperado | Motivo |
|---|---|---|
| `age` | 18 – 64 | Rango del estudio original |
| `bmi` | 10 – 60 | Límites fisiológicamente plausibles |
| `children` | 0 – 10 | Límite razonable |
| `charges` | > 0 | Un costo no puede ser negativo o cero |

In [ ]:
# ── Calidad del dato ───────────────────────────────────────────────────

# 1. Duplicados
n_duplicados = df.duplicated().sum()
print(f"Registros duplicados : {n_duplicados}")
if n_duplicados > 0:
    print("\nFilas duplicadas:")
    display(df[df.duplicated(keep=False)].sort_values(list(df.columns)))

# 2. Rangos inválidos
print("\n── Verificación de rangos lógicos ───────────────────────────────────")
rangos = {
    'age'     : (18,   64),
    'bmi'     : (10,   60),
    'children': (0,    10),
    'charges' : (0.01, None)   # solo límite inferior
}

resumen_rangos = []
for col, (minimo, maximo) in rangos.items():
    fuera_min = (df[col] < minimo).sum() if minimo is not None else 0
    fuera_max = (df[col] > maximo).sum() if maximo is not None else 0
    total_fuera = fuera_min + fuera_max
    resumen_rangos.append({
        'Variable'       : col,
        'Mín esperado'   : minimo,
        'Mín real'       : df[col].min(),
        'Máx esperado'   : maximo,
        'Máx real'       : df[col].max(),
        'Fuera de rango' : total_fuera,
        'Estado'         : '✅ OK' if total_fuera == 0 else '🔴 Error'
    })

display(pd.DataFrame(resumen_rangos).set_index('Variable'))

**💡 Qué hacer con cada hallazgo**

| Hallazgo | Acción recomendada |
|---|---|
| **Duplicados encontrados** | Eliminar con `df.drop_duplicates(inplace=True)` y documentar cuántos se removieron |
| **Sin duplicados** | Registrar como verificado ✅ |
| **Valores fuera de rango** | Investigar origen antes de eliminar — puede ser error de tipeo, unidad equivocada o problema de ingesta |
| **Todos en rango** | Registrar como verificado ✅ |

> En nuestro dataset no se esperan errores de rango —
> es un dataset curado para enseñanza. En datos reales de producción
> esta celda suele encontrar sorpresas.

### Detección de Outliers

Un **outlier** es un valor que se aleja significativamente del comportamiento
típico de la variable. A diferencia de un error de rango, puede ser un dato
**real y válido** — pero su presencia afecta los modelos de regresión lineal.

Usaremos dos métodos complementarios:

| Método | Basado en | Robusto ante sesgo | Cuándo usarlo |
|---|---|---|---|
| **IQR** | Percentiles Q1 y Q3 | ✅ Sí | Siempre — es el estándar |
| **Z-score** | Media y desviación estándar | ❌ No | Solo en distribuciones aproximadamente normales |

> Dado que `charges` tiene skewness **1.52**, el IQR será más confiable
> que el Z-score para esa variable. Los usaremos juntos para comparar.

#### Regla del IQR

$$\text{Límite inferior} = Q1 - 1.5 \times IQR$$
$$\text{Límite superior} = Q3 + 1.5 \times IQR$$

Todo valor fuera de ese rango es candidato a outlier.

Con los datos que ya conocemos de `charges`:
- Q1 = $4,740 · Q3 = $16,640 · IQR = $11,899
- Límite inferior = $4,740 − $17,849 = **−$13,109** → ningún valor puede caer aquí
- Límite superior = $16,640 + $17,849 = **$34,489** → valores sobre este umbral son outliers

In [ ]:
# ── Detección de outliers — Método IQR ────────────────────────────────
numericas = df.select_dtypes(include='number').columns.tolist()

resumen_iqr = []
for col in numericas:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lim_inf) | (df[col] > lim_sup)]
    n = len(outliers)

    resumen_iqr.append({
        'Variable'        : col,
        'Q1'              : Q1,
        'Q3'              : Q3,
        'IQR'             : IQR,
        'Límite inferior' : lim_inf,
        'Límite superior' : lim_sup,
        'N outliers'      : n,
        '% del total'     : round(100 * n / len(df), 2)
    })

df_iqr = pd.DataFrame(resumen_iqr).set_index('Variable')
display(df_iqr.style.format({
    'Q1': '{:.2f}', 'Q3': '{:.2f}', 'IQR': '{:.2f}',
    'Límite inferior': '{:.2f}', 'Límite superior': '{:.2f}',
    '% del total': '{:.2f}%'
}).background_gradient(subset=['N outliers'], cmap='OrRd'))

In [ ]:
# ── Boxplots con límites IQR anotados ─────────────────────────────────────
fig, axes = plt.subplots(1, len(numericas), figsize=(14, 5))

for ax, col in zip(axes, numericas):
    Q1      = df[col].quantile(0.25)
    Q3      = df[col].quantile(0.75)
    IQR     = Q3 - Q1
    lim_sup = Q3 + 1.5 * IQR
    lim_inf = Q1 - 1.5 * IQR

    sns.boxplot(data=df, y=col, ax=ax, color='steelblue',
                width=0.4,
                flierprops=dict(marker='o', markersize=4,
                                alpha=0.5, color='#E63946'))

    # Línea de límite superior si hay outliers arriba
    if df[col].max() > lim_sup:
        ax.axhline(lim_sup, color='#E63946', ls='--', lw=1.2,
                   label=f"Lím. sup: {lim_sup:.1f}")
        ax.legend(fontsize=8)

    n_out = ((df[col] < lim_inf) | (df[col] > lim_sup)).sum()
    ax.set_title(f"{col}\n({n_out} outliers IQR)", fontsize=11)

plt.suptitle("Detección de Outliers — Método IQR",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

#### Z-score

El Z-score estandariza cada valor expresándolo como número de
desviaciones estándar respecto a la media:

$$z_i = \frac{x_i - \bar{x}}{s}$$

**Regla estándar:** valores con |z| > 3 son outliers candidatos
(equivale a estar a más de 3 desviaciones estándar de la media).

> ⚠️ **Limitación importante:** como el Z-score usa la media y la
> desviación estándar, es sensible a los propios outliers que intenta
> detectar. En variables con sesgo fuerte como `charges` (skewness 1.52),
> puede **subestimar** la cantidad real de outliers.
> Compara siempre los resultados con IQR.

In [ ]:
# ── Detección de outliers — Método Z-score ────────────────────────────────
from scipy import stats

UMBRAL_Z = 3

resumen_z = []
for col in numericas:
    z_scores    = np.abs(stats.zscore(df[col]))
    n_outliers  = (z_scores > UMBRAL_Z).sum()
    idx_outliers = df.index[z_scores > UMBRAL_Z].tolist()

    resumen_z.append({
        'Variable'      : col,
        'Z máximo'      : z_scores.max(),
        'N outliers (Z)': n_outliers,
        '% del total'   : round(100 * n_outliers / len(df), 2),
        'N outliers IQR': df_iqr.loc[col, 'N outliers']
    })

df_z = pd.DataFrame(resumen_z).set_index('Variable')
display(df_z.style.format({
    'Z máximo': '{:.2f}',
    '% del total': '{:.2f}%'
}).background_gradient(subset=['N outliers (Z)'], cmap='OrRd'))

print("\n── Comparación IQR vs Z-score ───────────────────────────────────────")
print("Si IQR detecta más outliers que Z-score → distribución sesgada (esperado en 'charges')")
print("Si Z-score detecta más → distribución con colas pesadas simétricas")

In [ ]:
# ── Visualizar outliers de charges en contexto ────────────────────────────
# Colorear los outliers IQR dentro del scatterplot age vs charges

Q1_c      = df['charges'].quantile(0.25)
Q3_c      = df['charges'].quantile(0.75)
IQR_c     = Q3_c - Q1_c
lim_sup_c = Q3_c + 1.5 * IQR_c

df['es_outlier'] = df['charges'] > lim_sup_c

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter age vs charges
sns.scatterplot(
    data=df, x='age', y='charges', hue='es_outlier',
    palette={True: '#E63946', False: '#ADB5BD'},
    alpha=0.7, edgecolor='white', linewidth=0.3,
    ax=axes[0]
)
axes[0].axhline(lim_sup_c, color='#E63946', ls='--', lw=1.2,
                label=f"Lím. IQR: ${lim_sup_c:,.0f}")
axes[0].set_title("Outliers de charges — age vs charges")
axes[0].legend(fontsize=9)
axes[0].yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))

# Scatter bmi vs charges
sns.scatterplot(
    data=df, x='bmi', y='charges', hue='es_outlier',
    palette={True: '#E63946', False: '#ADB5BD'},
    alpha=0.7, edgecolor='white', linewidth=0.3,
    ax=axes[1]
)
axes[1].axhline(lim_sup_c, color='#E63946', ls='--', lw=1.2,
                label=f"Lím. IQR: ${lim_sup_c:,.0f}")
axes[1].set_title("Outliers de charges — bmi vs charges")
axes[1].legend(fontsize=9)
axes[1].yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))

plt.suptitle("Distribución espacial de outliers en charges",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Limpiar columna auxiliar
df.drop(columns=['es_outlier'], inplace=True)

#### Interpretación — ¿Qué hacer con los outliers?

Los outliers de `charges` no son errores — son pacientes reales con
costos médicos muy altos. La pregunta correcta no es *"¿los elimino?"*
sino *"¿qué los explica?"*

Al colorear por outlier en los scatterplots se hace evidente que:
- Los outliers de `charges` se concentran en **fumadores con BMI alto**
- No están distribuidos al azar → tienen una explicación estructural

| Opción | Cuándo usarla | Aplicación aquí |
|---|---|---|
| **Eliminar** | Error de datos confirmado | ❌ No aplica |
| **Transformar** (`log1p`) | Outliers reales que sesgan el modelo | ✅ Recomendado para `charges` |
| **Mantener con flag** | Outliers informativos que se quieren modelar | ✅ Alternativa válida |
| **Modelo robusto** | Cuando los outliers son la norma del negocio | Para sesiones avanzadas |

> **Decisión para la siguiente clase:** usaremos `np.log1p(charges)`
> como variable objetivo en regresión lineal. Compararemos los
> coeficientes con y sin transformación.

### ✅ Síntesis — Calidad y Anomalías

| Verificación | Resultado | Acción |
|---|---|---|
| Duplicados | Por confirmar en ejecución | Eliminar si > 0 |
| Rangos inválidos | Por confirmar en ejecución | Investigar origen |
| Outliers `age` | Sin outliers IQR esperados | Ninguna |
| Outliers `bmi` | Outliers leves por arriba | Monitorear |
| Outliers `charges` | Outliers reales (fumadores + BMI alto) | Aplicar `log1p` |
| Outliers `children` | Sin outliers IQR esperados | Ninguna |

**El dataset está en condiciones de ser usado para regresión lineal.**
Los pasos de preprocesamiento pendientes para la próxima sesión son:

1. `np.log1p(df['charges'])` → nueva columna `log_charges`
2. `pd.get_dummies(df, columns=['sex', 'smoker', 'region'], drop_first=True)`
3. Estandarización de `age` y `bmi` con `StandardScaler`

### 🔧 Tu turno

1. El dataset tiene **1 registro duplicado** (dato conocido del paper original).
   Encuéntralo, muéstralo y elimínalo. ¿Cambia alguna estadística descriptiva
   de forma notable?

2. Aplica `np.log1p()` a `charges` y vuelve a calcular el Z-score.
   ¿Cuántos outliers detecta ahora? ¿Por qué cambia el número?

3. Crea una función reutilizable `reporte_outliers(df, col)` que reciba
   un DataFrame y el nombre de una columna, y devuelva un diccionario con:
   - límites IQR
   - cantidad de outliers IQR
   - cantidad de outliers Z-score
   - recomendación automática basada en el nivel de skewness

In [ ]:
# Pon tú código aquí


## 8. Síntesis y Cierre del Taller

En esta sección consolidamos todo lo que descubrimos durante el análisis
y lo traducimos en decisiones concretas para la siguiente sesión.

Un EDA no termina cuando ejecutas la última celda —
termina cuando puedes responder estas tres preguntas:

| Pregunta | Estado |
|---|---|
| ¿Entiendo la estructura del dataset? | Por completar ↓ |
| ¿Identifiqué problemas de calidad? | Por completar ↓ |
| ¿Tengo hipótesis sobre qué variables predicen el target? | Por completar ↓ |

In [ ]:
# ── Reporte automático de EDA ──────────────────────────────────────────────
try:
    from ydata_profiling import ProfileReport

    reporte = ProfileReport(
        df,
        title="EDA — Medical Insurance Costs",
        explorative=True
        # dark_mode=False  # Removed as it's no longer a valid parameter
    )

    # Genera un HTML
    reporte.to_file("reporte_eda.html")
    print("✅ Reporte guardado en 'reporte_eda.html'")

    # Embeber en Jupyter
    reporte.to_notebook_iframe()

except ImportError:
    print("⚠️  ydata-profiling no está instalado.")
    print("   Instálalo con:  pip install ydata-profiling")
    print("\n── Resumen alternativo ──────────────────────────────────────")
    display(df.describe(include='all'))

### 💡 Reporte automático vs. análisis manual

El reporte de `ydata-profiling` hace en segundos lo que acabamos de
construir celda por celda durante 3 horas. ¿Para qué aprender el proceso manual?

| | Análisis manual | ydata-profiling |
|---|---|---|
| **Velocidad** | Lento | Instantáneo |
| **Comprensión** | ✅ Profunda | Superficial |
| **Decisiones** | ✅ Fundamentadas | Requiere interpretación |
| **Datasets grandes** | Lento | ✅ Eficiente |
| **Personalización** | ✅ Total | Limitada |
| **Detecta interacciones** | ✅ Con criterio | Parcialmente |

> **Regla práctica:** usa `ydata-profiling` para un primer vistazo rápido
> o para documentar. Usa el análisis manual cuando las decisiones de
> preprocesamiento importen — que es siempre antes de modelar.

### Mis Hallazgos — Tabla de Cierre

*Completa esta tabla con lo que observaste durante el taller.*

| Variable | Distribución | Outliers | Hallazgo principal | Acción aplicada |
|---|---|---|---|---|
| `age` | | | | |
| `bmi` | | | | |
| `charges` | | | | |
| `children` | | | | |
| `sex` | | | | |
| `smoker` | | | | |
| `region` | | | | |

**Mi hipótesis sobre el modelo de regresión:**

La variable que más influye en `charges` es _________________ porque _________________.

La transformación más importante antes de modelar es _________________ porque _________________.

Una relación que me sorprendió durante el análisis fue _________________.

### Puente hacia Regresión Lineal

Todo lo que descubrimos hoy tiene una traducción directa al modelo:

| Hallazgo EDA | Decisión en regresión |
|---|---|
| `charges` tiene skewness 1.52 | Usar `log_charges` como variable objetivo |
| `smoker` separa el dataset en dos bandas | Incluir como dummy — será el coeficiente más alto |
| `age` y `bmi` tienen relación positiva con `charges` | Incluir como predictores continuos estandarizados |
| Bimodalidad en fumadores con BMI alto | Probar término de interacción `smoker_yes × bmi` |
| `sex` sin diferencia significativa | Incluir pero esperar coeficiente cercano a 0 |
| `region` con efecto pequeño | Incluir para no omitir varianza explicable |

**Ecuación que construiremos la próxima sesión:**

$$\log(\text{charges}+1) = \beta_0 + \beta_1 \cdot age + \beta_2 \cdot bmi + \beta_3 \cdot children + \beta_4 \cdot smoker\_yes + \beta_5 \cdot sex\_male + \beta_6 \cdot region\_* + \varepsilon$$

**Preguntas que responderemos:**
- ¿Qué tan bien explica el modelo la varianza de `charges`? → R²
- ¿Los coeficientes coinciden con las hipótesis del EDA? → Interpretación de β
- ¿Se cumplen los supuestos de regresión lineal? → Diagnóstico de residuos

---

## ✅ Cierre

### Lo que hiciste hoy

| Sección | Técnica | Herramienta |
|---|---|---|
| Carga e inspección | `head`, `info`, `describe`, `dtypes` | pandas |
| Univariado categórico | Frecuencias, proporciones, desbalances | pandas · seaborn |
| Univariado numérico | Histogramas, boxplots, skewness, IQR | pandas · seaborn · scipy |
| Bivariado Num×Num | Correlación Pearson/Spearman, scatterplots | pandas · seaborn |
| Bivariado Num×Cat | Boxplots agrupados, violinplots, t-test | seaborn · scipy |
| Bivariado Cat×Cat | Tablas de contingencia, chi² | pandas · scipy |
| Calidad del dato | Duplicados, rangos inválidos | pandas |
| Outliers | IQR, Z-score, visualización en contexto | pandas · scipy |
| Síntesis | Reporte automático, decisiones de preprocesamiento | ydata-profiling · sklearn |

### Recursos para profundizar

- 📖 *Python for Data Analysis* — Wes McKinney (creador de pandas)
- 📖 *Storytelling with Data* — Cole Nussbaumer Knaflic
- 🔗 [Pandas documentation](https://pandas.pydata.org/docs/)
- 🔗 [Seaborn gallery](https://seaborn.pydata.org/examples/index.html)
- 🔗 [ydata-profiling docs](https://docs.profiling.ydata.ai)

---
*Próxima sesión: Regresión Lineal — usaremos `df_modelo` que preparamos hoy.*